# Notebook 2: Build a BPE Tokenizer from Scratch

**Goal**: Train a Byte-Pair Encoding tokenizer on the running corpus and develop intuition for how tokenization works.

After this notebook you will:
- Understand BPE merge rules and why they work
- Know how to encode/decode text
- Have intuition for vocabulary size tradeoffs

In [ ]:
import sys
sys.path.insert(0, '..')

from 01_data.generate_synthetic import SyntheticRunGenerator
from 02_tokenization.bpe_tokenizer import BPETokenizer

gen = SyntheticRunGenerator(seed=42)
corpus = gen.generate_run_notes(2000)
print(f'Corpus: {len(corpus):,} documents')
print(f'Sample: {corpus[0]}')

## 1. Train a small tokenizer (watch the merges happen)

In [ ]:
# Start tiny so you can watch every merge
tok_small = BPETokenizer(vocab_size=120)
tok_small.train(corpus[:200], verbose=True)

print(f'\nFirst 20 merges:')
for i, (a, b) in enumerate(tok_small.merges[:20]):
    print(f'  {i+1:2d}. ({repr(a)}, {repr(b)}) -> {repr(a+b)}')

## 2. Train the full tokenizer for model training

In [ ]:
tok = BPETokenizer(vocab_size=500)   # small for demo; use 8000 for real training
tok.train(corpus, verbose=True)

print(f'\nVocab size: {len(tok):,}')

## 3. Encode / Decode roundtrip

In [ ]:
test_texts = [
    "easy run 6 miles at 9:30 pace",
    "marathon training long run 18 miles felt strong",
    "5K race personal best 24:15 sunny weather",
]

for text in test_texts:
    ids = tok.encode(text)
    decoded = tok.decode(ids)
    tokens = [tok.vocab[i] for i in ids]
    print(f'Original:  {text}')
    print(f'Tokens:    {tokens}')
    print(f'IDs:       {ids}')
    print(f'Decoded:   {decoded}')
    print(f'Compression: {len(text)} chars -> {len(ids)} tokens ({len(ids)/len(text):.2f} tokens/char)')
    print()

## 4. Vocabulary analysis — what did BPE learn?

In [ ]:
# Separate single chars from multi-char tokens
single = [(i, t) for i, t in tok.vocab.items() if len(t) == 1 and i >= 4]
multi = [(i, t) for i, t in tok.vocab.items() if len(t) > 1 and not t.startswith('<|')]
multi_sorted = sorted(multi, key=lambda x: len(x[1]), reverse=True)

print(f'Single-char tokens: {len(single)}')
print(f'Multi-char tokens:  {len(multi)}')
print()
print('Longest tokens (most compressed):')
for tid, tok_str in multi_sorted[:20]:
    print(f'  ID {tid:5d}: {repr(tok_str)}')

In [ ]:
# Token length distribution
import matplotlib.pyplot as plt

token_lengths = [len(t) for t in tok.vocab.values() if not t.startswith('<|')]
plt.figure(figsize=(8, 4))
plt.hist(token_lengths, bins=range(1, max(token_lengths)+2), color='steelblue', edgecolor='white')
plt.xlabel('Token length (characters)')
plt.ylabel('Count')
plt.title('BPE token length distribution — running corpus')
plt.show()
print(f'Mean token length: {sum(token_lengths)/len(token_lengths):.2f} chars')

## 5. Vocab size tradeoff experiment

In [ ]:
test_sentence = "marathon training 18 mile long run at 9:45 pace avg heart rate 148 bpm elevation gain 650 feet"

results = []
for vocab_size in [50, 100, 200, 500]:
    t = BPETokenizer(vocab_size=vocab_size)
    t.train(corpus[:500], verbose=False)
    ids = t.encode(test_sentence)
    results.append((vocab_size, len(ids)))
    print(f'vocab={vocab_size:5d}: {len(ids):3d} tokens  {[t.vocab[i] for i in ids]}')

## 6. Special tokens

Understanding BOS/EOS/PAD tokens — critical for training.

In [ ]:
print('Special token IDs:')
print(f'  PAD: {tok.pad_id}  -> {repr(tok.vocab[tok.pad_id])}')
print(f'  BOS: {tok.bos_id}  -> {repr(tok.vocab[tok.bos_id])}')
print(f'  EOS: {tok.eos_id}  -> {repr(tok.vocab[tok.eos_id])}')
print()

text = "easy run 5 miles"
ids_with = tok.encode(text, add_special=True)
ids_without = tok.encode(text, add_special=False)
print(f'Without specials: {ids_without}')
print(f'With specials:    {ids_with}  (BOS={tok.bos_id} prepended, EOS={tok.eos_id} appended)')

## Exercise

1. Find the token IDs for 'marathon', 'pace', '5K'. Are they single tokens or split?
2. Encode '3:45:22' (a finish time). How many tokens? Which merge rules fired?
3. What happens when you encode text in a different language? Try a Spanish running phrase.
4. Save your tokenizer: `tok.save('../02_tokenization/tokenizer_demo.json')` and reload it.